# W&B Runtime against the HF billed usage
This analysis only reads the CSV the user provided. The 12.57 HF hours come from a chat screenshot and are the rounded billed hardware usage.
Created + Runtime is only an estimate and cannot replace an independent stop time; crashed is not enough to tell why HF stopped. The CSV has no HF timeout or Job ID, and no training history metrics.

In [1]:
from pathlib import Path
import csv, json
from datetime import datetime, timedelta, timezone
source = Path("/Users/ethan/Downloads/wandb_export_2026-10-04T11_40_46.724+08_00.csv")
with source.open(encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    columns = reader.fieldnames
    rows = list(reader)
assert len(rows) == 1
row = rows[0]
seconds = int(row["Runtime"])
created = datetime.fromisoformat(row["Created"].replace("Z", "+00:00"))
hk = timezone(timedelta(hours=8))
result = {
    "rows": len(rows), "columns": len(columns),
    "duplicate_headers": len(columns) - len(set(columns)),
    "ragged_rows": sum(None in r or any(v is None for v in r.values()) for r in rows),
    "name": row["Name"], "state": row["State"],
    "created_hk": created.astimezone(hk).isoformat(),
    "runtime_seconds": seconds, "runtime_hours": seconds / 3600,
    "runtime_hms": str(timedelta(seconds=seconds)),
    "inferred_end_if_elapsed_runtime_hk": (created + timedelta(seconds=seconds)).astimezone(hk).isoformat(),
    "bill_display_hours": 12.57,
    "bill_display_minus_runtime_seconds": round(12.57 * 3600 - seconds),
    "non_env_config_columns": [c for c in columns if not c.startswith("env_cfg.")]
}
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "rows": 1,
  "columns": 500,
  "duplicate_headers": 0,
  "ragged_rows": 0,
  "name": "2026-10-03_13-14-58_md-first-s42-20261003-205907",
  "state": "crashed",
  "created_hk": "2026-10-03T21:16:07+08:00",
  "runtime_seconds": 44985,
  "runtime_hours": 12.495833333333334,
  "runtime_hms": "12:29:45",
  "inferred_end_if_elapsed_runtime_hk": "2026-10-04T09:45:52+08:00",
  "bill_display_hours": 12.57,
  "bill_display_minus_runtime_seconds": 267,
  "non_env_config_columns": [
    "Name",
    "State",
    "Notes",
    "User",
    "Tags",
    "Created",
    "Runtime",
    "Sweep"
  ]
}
